In [1]:
!conda env create --name q2-fungal-traits-moshpit-2026.10 --file https://raw.githubusercontent.com/bokulich-lab/q2-fungal-traits/refs/heads/main/environment-files/q2-fungal-traits-rachis-moshpit-2026.10.yml
!conda run --no-capture-output -n q2-fungal-traits-moshpit-2026.10 pip install "q2-fungal-traits@git+https://github.com/bokulich-lab/q2-fungal-traits.git"
!conda run --no-capture-output -n q2-fungal-traits-moshpit-2026.10 qiime dev refresh-cache

Error loading anaconda-anon-usage: module 'conda.cli.install' has no attribute 'check_prefix'
Retrieving notices: done
Channels:
 - https://packages.qiime2.org/qiime2/2026.10/moshpit/released
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/conda/envs/q2-fungal-traits-moshpit-2026.10

  added / updated specs:
    - pip
    - rachis-moshpit


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libpython-3.12.15          |ha57cdbe_2_cpython         8.4 MB  conda-forge
    python-3.12.14             |h8ab3286_1_cpython        30.1 MB  conda-forge
    ------------------------------------------------------------
                                           Total:        38.5 MB

The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-8_kmp_llvm 
  _python_abi3_sup

In [2]:
!conda run --no-capture-output -n q2-fungal-traits-moshpit-2026.10 qiime fungal-traits annotate --help

Usage: qiime fungal-traits annotate [OPTIONS]

  Annotate taxonomy data with fungal lifestyle traits and spore volume
  metadata. Fungal trait annotation is performed when both genus and phylum
  are present, because the trait table is matched by genus and then filtered
  so only rows with the same phylum are retained. Spore volume annotation is
  performed when kingdom is present together with at least one of species,
  genus, or family. Spore matching is done in a simple fallback order: first
  try an exact species match, if that is not available try genus, and if that
  is still not available try family. Species matches use the direct spore
  volume from the source table. Genus and family matches summarize the
  available values for that rank with a geometric mean and record which rank
  was used, so it is clear whether a value came from a species, genus, or
  family-level match.

Inputs:
  --i-taxonomy ARTIFACT FeatureData[Taxonomy]
                         Fungal taxonomy.        

In [6]:
import os

ft_env   = "q2-fungal-traits-moshpit-2026.10"
in_dir_1 = "/home/jovyan/FunGut/data/processed"
in_dir_2 = "/home/jovyan/FunGut/data/processed_ignored"
out_dir_1  = "/home/jovyan/FunGut/data/processed/Fungal_Annotation"
out_dir_2  = "/home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation"

os.makedirs(out_dir_1, exist_ok=True)
os.makedirs(out_dir_2, exist_ok=True)

In [11]:
!conda run --no-capture-output -n $ft_env \
  qiime fungal-traits annotate \
  --i-taxonomy "$in_dir_2/03_taxonomy/taxonomy.qza" \
  --o-fungal-traits "$out_dir_2/fungal_traits.qza"

Saved ImmutableMetadata to: /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/fungal_traits.qza


In [12]:
!conda run --no-capture-output -n $ft_env \
  qiime metadata tabulate \
  --m-input-file "$out_dir_2/fungal_traits.qza" \
  --o-visualization "$out_dir_1/fungal_traits.qzv"

Saved Visualization to: /home/jovyan/FunGut/data/processed/Fungal_Annotation/fungal_traits.qzv


In [13]:
!conda run --no-capture-output -n $ft_env \
  qiime tools export \
  --input-path "$out_dir_2/fungal_traits.qza" \
  --output-path "$out_dir_2/fungal_traits_tsv"

Exported /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/fungal_traits.qza as ImmutableMetadataDirectoryFormat to directory /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/fungal_traits_tsv


In [14]:
import pandas as pd

traits = pd.read_csv(f"{out_dir_2}/fungal_traits_tsv/metadata.tsv", sep="\t", index_col=0)
traits = traits[~traits.index.astype(str).str.startswith("#")]

print("Features in total:", len(traits))
for col in ["fruitbody_type_template", "primary_lifestyle", "growth_form_template"]:
    print("\n=====", col)
    print(traits[col].value_counts(dropna=False))

Features in total: 681

===== fruitbody_type_template
fruitbody_type_template
none                                             317
NaN                                              293
perithecium_(hymenium_hidden,_narrow_opening)     35
agaricoid                                         11
cleistothecium_(closed,_spherical)                 7
polyporoid                                         5
corticioid                                         5
apothecium_(hymenium_on_surface)                   4
other                                              2
clavarioid                                         1
cyphelloid                                         1
Name: count, dtype: int64

===== primary_lifestyle
primary_lifestyle
NaN                       190
nectar/tap_saprotroph     177
unspecified_saprotroph     93
litter_saprotroph          54
soil_saprotroph            43
animal_parasite            40
plant_pathogen             37
wood_saprotroph            26
dung_saprotroph             6
